In [1]:
from src.dm import DataModule

dm = DataModule(batch_size=4, num_workers=0, pin_memory=False)

dm.setup()

In [2]:
dm.test_df

,image,AOD
0,data/test_images/test_1.tif,1.442470
1,data/test_images/test_2.tif,0.910374
2,data/test_images/test_3.tif,0.500745
3,data/test_images/test_4.tif,0.999882
4,data/test_images/test_5.tif,1.668206
...,...,...
1484,data/test_images/test_1485.tif,1.597977
1485,data/test_images/test_1486.tif,1.819262
1486,data/test_images/test_1487.tif,1.334963
1487,data/test_images/test_1488.tif,0.231196


In [3]:
import os

os.listdir('checkpoints')

['baseline-val_metric=0.83894-epoch=99.ckpt',
 'lrsch-val_metric=0.89440-epoch=492.ckpt',
 'da-epoch=199.ckpt',
 'lrsch-epoch=199.ckpt',
 'lrsch-epoch=499-v1.ckpt',
 'lrsch-val_metric=0.94095-epoch=461.ckpt',
 'baseline-epoch=199.ckpt',
 'baseline-epoch=99.ckpt',
 'baseline-val_metric=0.92066-epoch=199.ckpt',
 'lrsch-val_metric=0.86534-epoch=96.ckpt',
 'da-epoch=99.ckpt',
 'lrsch-epoch=499.ckpt',
 'ft-val_metric=0.84048-epoch=99.ckpt',
 'da-val_metric=0.82154-epoch=99.ckpt',
 'lrsch-epoch=99.ckpt',
 'ft-epoch=99.ckpt',
 'lrsch-val_metric=0.92938-epoch=188.ckpt',
 'da-val_metric=0.91423-epoch=199.ckpt']

In [9]:
import torch 
from src.module import Module

name = "lrsch-val_metric=0.94095-epoch=461.ckpt"
checkpoint = f'./checkpoints/{name}'


checkpoint = torch.load(checkpoint, map_location='cpu')
state_dict = checkpoint['state_dict']
module = Module()
module.load_state_dict(state_dict)

<All keys matched successfully>

In [13]:
checkpoint['hyper_parameters']

{'freeze': True,
 'optimizer': 'Adam',
 'optimizer_params': {'lr': 0.0003},
 'ckpt_path': None,
 'load_from_checkpoint': None,
 'trainer': {'accelerator': 'cuda',
  'devices': 1,
  'max_epochs': 500,
  'logger': <lightning.pytorch.loggers.wandb.WandbLogger at 0x7287e0225d60>,
  'enable_checkpointing': True,
  'overfit_batches': 0,
  'precision': '16-mixed',
  'deterministic': True,
  'callbacks': [<lightning.pytorch.callbacks.model_checkpoint.ModelCheckpoint at 0x7287e02259d0>,
   <lightning.pytorch.callbacks.model_summary.ModelSummary at 0x7287a20e9730>]},
 'datamodule': {'batch_size': 64,
  'num_workers': 20,
  'pin_memory': True,
  'train_trans': {},
  'val_size': 0.2,
  'bands': [2, 3, 4, 5, 6, 7, 8, 9, 11, 12]},
 'scheduler': 'OneCycleLR',
 'scheduler_params': {'max_lr': 0.001,
  'total_steps': 500,
  'pct_start': 0.03,
  'final_div_factor': 10,
  'verbose': True}}

In [14]:
from tqdm import tqdm

device = "cuda:1"
module.to(device)
module.eval()

preds = []
for batch in tqdm(dm.test_dataloader()):
	dict_of_tensors, _ = batch
	dict_of_tensors = {k: v.to(device) for k, v in dict_of_tensors.items()}
	with torch.no_grad():
		output = module(dict_of_tensors)
		output = output * dm.aod_stats[1] + dm.aod_stats[0] 
		preds += output.cpu().tolist()

100%|██████████| 373/373 [00:28<00:00, 12.89it/s]


In [15]:
dm.test_df['AOD'] = preds
dm.test_df.image = dm.test_df.image.apply(lambda x: x.split('/')[-1])

dm.test_df

,image,AOD
0,test_1.tif,0.062797
1,test_2.tif,0.230022
2,test_3.tif,0.202643
3,test_4.tif,0.082800
4,test_5.tif,0.088936
...,...,...
1484,test_1485.tif,0.139626
1485,test_1486.tif,0.128737
1486,test_1487.tif,0.036608
1487,test_1488.tif,0.158085


In [16]:
dm.test_df.to_csv('submission.csv', index=False, header=False)